# Task 5 — LoRA nell'encoder di SAM (Kaggle, GPU)

LoRA su q e v dell'attenzione dei 12 blocchi dell'encoder (r = 4, 8, 16) + mask decoder.
Nello stesso notebook rifacciamo il **solo decoder con lo stesso protocollo** (stesse epoche, stessa precisione, stessa selezione), così il confronto è alla pari.

**Session options:** *Accelerator* **GPU T4 x2** (ne usiamo una) · *Internet* **On** · *Persistence* **Files only**.

**Come eseguirlo**
1. Esegui le sezioni 1–4 (fino al benchmark) e leggi i tempi.
2. Imposta la sezione 5 (epoche, AMP, rank) in base al benchmark.
3. Esegui il resto: *Run → Run after* sulla cella della sezione 5 (o *Save Version → Save & Run All* per lavorare in background con i valori impostati).

Ogni run, se ha già i risultati su disco, viene saltato: se la sessione si interrompe, rieseguendo il notebook si riparte dal run mancante.

## 1. Codice della repo

In [ ]:
from pathlib import Path
import sys

REPO_URL = "https://github.com/caMatt99/sam-peft-kvasir.git"
WORK = Path("/kaggle/working")
REPO = WORK / "sam-peft-kvasir"

if REPO.exists():
    !cd {REPO} && git pull -q
else:
    !git clone -q {REPO_URL} {REPO}
!pip install -q -e "{REPO}[dev]"
%cd {REPO}
sys.path.insert(0, str(REPO / "src"))   # il kernel già avviato non vede l'install editable
!git log --oneline -1

## 2. Dati e pesi

In [ ]:
import torch
assert torch.cuda.is_available(), "Accendi la GPU in Session options → Accelerator"
print(torch.cuda.get_device_name(0))

def find_kvasir_root():
    for base in [Path("/kaggle/input"), WORK / "data"]:
        if base.exists():
            for p in base.rglob("images"):
                if (p.parent / "masks").is_dir():
                    return p.parent
    return None

RAW = find_kvasir_root()
if RAW is None:
    (WORK / "data").mkdir(parents=True, exist_ok=True)
    !wget -q -L --no-check-certificate https://datasets.simula.no/downloads/kvasir-seg.zip -O /kaggle/working/data/kvasir-seg.zip
    !unzip -q /kaggle/working/data/kvasir-seg.zip -d /kaggle/working/data/ && rm /kaggle/working/data/kvasir-seg.zip
    RAW = find_kvasir_root()
assert RAW is not None

PROCESSED = WORK / "processed"
if not (PROCESSED / "meta.json").exists():
    !python scripts/prepare_data.py --raw-root {RAW} --out {PROCESSED} > /dev/null

CKPT = WORK / "checkpoints" / "sam_vit_b_01ec64.pth"
if not CKPT.exists() or CKPT.stat().st_size < 300e6:
    CKPT.parent.mkdir(parents=True, exist_ok=True)
    !wget -q -L https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O {CKPT}

CACHE = WORK / "cache" / "vit_b"
COMMON = f"--raw-root {RAW} --processed {PROCESSED} --checkpoint {CKPT}"
done = lambda run: Path(f"{run}/test/metrics.json").exists()
print(RAW, "|", PROCESSED, "| checkpoint MB:", round(CKPT.stat().st_size / 1e6))

## 3. Test
Nuovi in questa task (`tests/test_lora.py`): all'inizio LoRA coincide con SAM (B = 0), conta dei parametri, solo A, B e decoder si addestrano, il merge dà le stesse uscite, lo script di training gira con LoRA.

In [ ]:
!python -m pytest -q

## 4. Benchmark
LoRA r = 8 su 48 immagini di training (dopo 8 di riscaldamento) e 20 di validation, in float32 e in AMP (float16).
Se una modalità esaurisce la memoria della T4, viene ripetuta con il *gradient checkpointing* (meno memoria, ~30% più lento).
Durata: ~5 minuti. **Annota la tabella: entra nel diario.**

In [ ]:
!python scripts/benchmark.py {COMMON} --out runs/benchmark.json

## 5. Protocollo comune — DA IMPOSTARE dopo il benchmark
Valgono per **tutti** i run di questo notebook (decoder e LoRA) e, più avanti, per il prompt tuning.
- `EPOCHS`: la curva della Task 2 mostra che il decoder converge in ~8 epoche.
- `AMP`: float16 nel training se è nettamente più veloce (la valutazione finale resta in float32).
- `CHECKPOINTING`: True solo se il benchmark ha dovuto usarlo per la modalità scelta.
- `RANKS`: se il tempo non basta si riducono le epoche, non i rank.

In [ ]:
EPOCHS = 10
AMP = True
CHECKPOINTING = False
RANKS = [4, 8, 16]

FLAGS = f"--epochs {EPOCHS}" + (" --amp" if AMP else "") + (" --checkpointing" if CHECKPOINTING else "")
RUNS = {"decoder": "runs/decoder", **{f"lora r={r}": f"runs/lora_r{r}" for r in RANKS}}
print(FLAGS); print(RUNS)

## 6. Zero-shot sul test
Righe di riferimento per le tabelle (≈2 minuti).

In [ ]:
for flag, name in [("", "zeroshot"), ("--multimask", "zeroshot_multimask")]:
    if not done(f"runs/{name}"):
        !python scripts/evaluate.py {COMMON} --split test {flag} --out runs/{name}

## 7. Solo decoder, protocollo comune
Cache degli embedding (~7 min, saltata se già presente) e training. **Controllo:** epoca 0 ≈ 0.906.

In [ ]:
run = RUNS["decoder"]
if not done(run):
    !python scripts/cache_embeddings.py {COMMON} --out {CACHE}
    !python scripts/train.py {COMMON} --method decoder --cache {CACHE} {FLAGS} --out {run}
    for s in ["val", "test"]:
        !python scripts/evaluate.py {COMMON} --split {s} --weights {run}/trainable.pt --out {run}
else:
    print("già fatto:", run)

## 8. LoRA
Un run per rank. L'encoder gira a ogni passo: è qui che si spende il tempo.
**Controllo:** epoca 0 ≈ 0.906 per ogni rank (B = 0, quindi all'inizio è SAM originale).
La valutazione finale fa il *merge* di LoRA nei pesi: l'inferenza costa come SAM.

In [ ]:
for r in RANKS:
    run = RUNS[f"lora r={r}"]
    if done(run):
        print("già fatto:", run); continue
    print(f"\n========== LoRA r = {r} ==========")
    !python scripts/train.py {COMMON} --method lora --rank {r} {FLAGS} --out {run}
    for s in ["val", "test"]:
        !python scripts/evaluate.py {COMMON} --split {s} --weights {run}/trainable.pt --out {run}

## 9. Analisi
### Tabella dei metodi (test)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

load = lambda run, split_name="test": json.loads(Path(f"{run}/{split_name}/metrics.json").read_text())
summ = {name: json.loads(Path(f"{run}/summary.json").read_text()) for name, run in RUNS.items()}

def fmt(g, k):
    return f"{g[f'{k}.dice_raw']['mean']:.3f} ± {g[f'{k}.dice_raw']['sem']:.3f}"

rows = []
for name, run in {"zero-shot": "runs/zeroshot", "zero-shot 3 maschere": "runs/zeroshot_multimask", **RUNS}.items():
    m = load(run); g = m["groups"]["all"]; s = summ.get(name)
    rows.append({
        "metodo": name,
        "param. encoder": s["parameters"]["encoder"] if s else 0,
        "param. addestrati": s["parameters"]["trainable"] if s else 0,
        "Dice": fmt(g, "boxes"), "Dice jitter": fmt(g, "boxes_jitter"),
        "fail": f"{g['boxes.fail_rate']:.0%}",
        "large_polyp": f"{m['groups']['large_polyp']['boxes.dice_raw']['mean']:.3f}" if "large_polyp" in m["groups"] else "-",
        "s / epoca": round(s["train_s_per_epoch_median"]) if s else None,
        "training (min)": round(s["train_s_total"] / 60, 1) if s else None,
        "memoria (GB)": round(s["peak_gpu_memory_mb"] / 1024, 1) if s and s["peak_gpu_memory_mb"] else None,
        "epoca migliore": s["best_epoch"] if s else None,
        "inferenza encoder (ms)": round(m["timing_median_ms"]["encoder_ms"]),
    })
table = pd.DataFrame(rows).set_index("metodo")
display(table)

### Confronto appaiato con il solo decoder (test)
È il confronto che risponde alla domanda del progetto: **adattare l'encoder con LoRA serve, rispetto ad addestrare solo il decoder?**

In [ ]:
per = lambda run: pd.read_csv(f"{run}/test/per_image.csv").set_index("id")
dec = per(RUNS["decoder"])
for r in RANKS:
    lo = per(RUNS[f"lora r={r}"]).loc[dec.index]
    for k in ["boxes", "boxes_jitter"]:
        diff = lo[f"{k}.dice_raw"] - dec[f"{k}.dice_raw"]
        print(f"LoRA r={r:<2} − decoder  {k:<13} {diff.mean():+.4f} ± {diff.std(ddof=1) / np.sqrt(len(diff)):.4f}"
              f"   migliora {(diff > 0.01).sum():3d} · peggiora {(diff < -0.01).sum():3d} · pari {(diff.abs() <= 0.01).sum():3d}")

### Curve di training

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 3.8))
for name, run in RUNS.items():
    h = pd.DataFrame(json.loads(Path(f"{run}/history.json").read_text()))
    ax[0].plot(h["epoch"][1:], h["train_loss"][1:], label=name)
    ax[1].plot(h["epoch"], h["val_dice"], label=name)
    ax[2].plot(h["epoch"], h["val_dice_jitter"], label=name)
for a, t in zip(ax, ["loss di training", "val Dice, box esatti", "val Dice, box con jitter"]):
    a.set_title(t); a.set_xlabel("epoca"); a.legend()
plt.show()
print("epoca 0 (deve essere ~0.906):", {n: round(json.loads(Path(f"{r}/history.json").read_text())[0]["val_dice"], 3)
                                        for n, r in RUNS.items()})

### Trade-off (anteprima della Task 6)
Dice sul test contro parametri adattati nell'encoder e contro tempo di training. Barre = errore standard.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for name, run in RUNS.items():
    g = load(run)["groups"]["all"]["boxes.dice_raw"]; s = summ[name]
    for a, x in zip(ax, [s["parameters"]["encoder"] / 1e3, s["train_s_total"] / 60]):
        a.errorbar(x, g["mean"], yerr=g["sem"], fmt="o", capsize=3)
        a.annotate(name, (x, g["mean"]), textcoords="offset points", xytext=(5, 5), fontsize=9)
for name, run, ls in [("zero-shot 3 maschere", "runs/zeroshot_multimask", ":")]:
    for a in ax:
        a.axhline(load(run)["groups"]["all"]["boxes.dice_raw"]["mean"], ls=ls, c="gray", label=name)
ax[0].set_xlabel("parametri adattati nell'encoder (migliaia)"); ax[1].set_xlabel("tempo di training (min)")
for a in ax: a.set_ylabel("Dice test (box esatti)"); a.legend()
plt.show()

### Per dimensione del polipo (test, box esatti)

In [ ]:
zs = per("runs/zeroshot")
tab = pd.DataFrame({"area": zs["area_frac"], "zero-shot": zs["boxes.dice_raw"],
                    **{n: per(r).loc[zs.index, "boxes.dice_raw"] for n, r in RUNS.items()}})
tab["bin"] = pd.cut(tab["area"], [0, .1, .25, .5, 1], labels=["<10%", "10-25%", "25-50%", ">50%"])
display(tab.groupby("bin", observed=True).agg(n=("area", "size"),
        **{c: (c, "mean") for c in ["zero-shot", *RUNS]}).round(3))

## 10. Salvataggio
Crea `/kaggle/working/task5_results.zip` (metriche, curve, benchmark, pesi addestrati; senza la cache).
Download: pannello **Output** → ↻ → `task5_results.zip` → ⋮ → **Download**. Con *Save & Run All* lo trovi nella scheda *Output* della versione.

In [ ]:
!rm -f {WORK}/task5_results.zip
!zip -qr {WORK}/task5_results.zip runs/
!ls -lh {WORK}/task5_results.zip

## Da annotare
1. Benchmark: secondi per immagine e memoria in float32 e AMP; è servito il checkpointing?
2. Epoca 0 ≈ 0.906 per tutti i run?
3. Decoder con il protocollo comune: quanto cambia rispetto alla Task 2 (0.956)?
4. LoRA − decoder, confronto appaiato: per quale rank (se c'è) il guadagno supera ~2 errori standard?
5. Il guadagno cresce con r? Dove sta, per dimensione del polipo e con il jitter?
6. Costo: s/epoca, tempo totale, memoria, inferenza (deve essere uguale a SAM grazie al merge).